# ADS v2: Motor renewal events

Builds 56 event families: `R0`, `R0.F1`, `R0.B1`, `R0.F2`, `R1a`, `R1b`, `R1.B2`, `R1.E1`, `R1.E1.F3`, `R2a`, `R2b`, `R2.F1`, `R2.B1.F1`, `R2.E1`, `R3a`, `R3b`, `R3c`, `R3d`, `R3b.F1`, `R3.O1`, `R4`, `R4.F1`, `R4a`, `R4c`, `R5`, `R6`, `R2.B1.1`, `R2.B1.2`, `R2.B1.3`, `R2.B1.3.F1`, `R2.B1.4.F1`, `R2.B1.O1`, `R2.B1.O2`, `R2.B1.O3`, `R3.B1`, `R3.B1.1`, `R3.B1.1.F1`, `R3.B1.2`, `R3.B1.2.F1`, `R3.B1.3`, `R3.B1.4`, `R3.B1.O1`, `R3.B1.O3`, `R6.F1`, `R4.B2.F1`, `R4.B2.1`, `R4.B2.2`, `R4.B2.4`, `R4.B2.5`, `R4.B2.3`, `R4.B2.O1`, `R4.B2.6`, `R6.B1`, `R6.B1.1`, `R6.B1.2`, `R6.B1.5`.

Generated by `ads_build/v2/build_ads.py` from `families/*.yaml`. Event descriptions come from `seeds/ChillEpisodeTypes.csv`.

**If a cell fails**

1. Read the failing cell's header: what it builds, what it reads, its grain, key and dedupe rule.
2. Fix the cause. Most failures are a source change (a renamed or missing column) or a broken key (duplicates).
   The checks cell at the end names the table and the likely cause.
3. Re-run from the failed cell: every cell rebuilds its table completely (`create or replace`), so re-running is
   always safe. After a session restart, run the first three cells again first (parameters, setup, start log).
4. **Copy the fix back into the config** (`ads_build/v2/families/*.yaml`) and regenerate. This notebook is generated:
   a fix made only here is lost at the next generation (the drift check lists such cells).


In [ ]:
%%pyspark
# Parameters (leave blank when running by hand)
batch_id = ""


In [ ]:
%%pyspark
%run ADS_v2_00_Setup

In [ ]:
%%pyspark
ads_run_log("12_Motor_Renewal", "started")


## R0: Motor Renewal - Pre-renewal notification, 60 days


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r0  (R0)                                                                */
/*  Description : Renewals / Spine. Notebook 04 cell 106. Every first-offer motor renewal, 60 days befo */
/*              : re the renew date (one per renewal). Change: events outside the build window are drop */
/*              : ped (04 had no lower bound).                                                          */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyRenewal                                */
/*              : Reads: ods_v2.PolicyRenewal                                                           */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r0 as
select  xxhash64('R0', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R0'                                                as EventTypeId,
        'Motor Renewal - Pre-renewal notification, 60 days' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(date_sub(LYPolicyRenewDateAdj, 60) as timestamp)           as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.ods}.PolicyRenewal
        where   PolicyOfferNum = 1
        and     RenewalMonth < date '${ads.end_time}'
        and     LYPolicyTypeGroup = 'Motor'
        and     cast(date_sub(LYPolicyRenewDateAdj, 60) as timestamp) >= timestamp '${ads.start_time}' and cast(date_sub(LYPolicyRenewDateAdj, 60) as timestamp) < timestamp '${ads.end_time}'
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R0.F1: Motor Renewal - Not sent - no terms from Insurer


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r0_f1  (R0.F1)                                                          */
/*  Description : Renewals / Friction. Notebook 04 cell 107. 'Renewal Offer % Held %' documents (no ter */
/*              : ms from the insurer), first per policy and month.                                     */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyDocumentEvent                          */
/*              : Reads: stg.adsv2_relay_document_event                                                 */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r0_f1 as
select  xxhash64('R0.F1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R0.F1'                                             as EventTypeId,
        'Motor Renewal - Not sent - no terms from Insurer' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(EventDateTime)                                              as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.stg}.adsv2_relay_document_event
        where   PolicyTypeGroup = 'Motor'
        and     EventDescription like 'Renewal Offer % Held %'
        and     EventDateTime >= timestamp '${ads.start_time}' and EventDateTime < timestamp '${ads.end_time}'
        group by PolicyCode, date_trunc('MONTH', EventDateTime)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R0.B1: Motor Renewal - Renewal held letter


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r0_b1  (R0.B1)                                                          */
/*  Description : Renewals / Branch. Notebook 04 cell 108. The same rule as R0.F1 (as today; 04 flags i */
/*              : t for confirmation).                                                                  */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyDocumentEvent                          */
/*              : Reads: stg.adsv2_relay_document_event                                                 */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r0_b1 as
select  xxhash64('R0.B1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R0.B1'                                             as EventTypeId,
        'Motor Renewal - Renewal held letter' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(EventDateTime)                                              as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.stg}.adsv2_relay_document_event
        where   PolicyTypeGroup = 'Motor'
        and     EventDescription like 'Renewal Offer % Held %'
        and     EventDateTime >= timestamp '${ads.start_time}' and EventDateTime < timestamp '${ads.end_time}'
        group by PolicyCode, date_trunc('MONTH', EventDateTime)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R0.F2: Motor Renewal - Declared claims not recorded


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r0_f2  (R0.F2)                                                          */
/*  Description : Renewals / Friction. Notebook 04 cell 109. Own-fault motor claims on renewing policie */
/*              : s, latest per policy and month.                                                       */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyClaimMotor, PolicyRenewal              */
/*              : Reads: ods_v2.PolicyClaimMotor, stg.adsv2_renewal_base                                */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r0_f2 as
select  xxhash64('R0.F2', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R0.F2'                                             as EventTypeId,
        'Motor Renewal - Declared claims not recorded' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(max(c.ClaimDate) as timestamp)                             as EventDateTime,
                1                                                               as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.ods}.PolicyClaimMotor c
        join    ${ads.stg}.adsv2_renewal_base b on b.PolicyCode = c.PolicyCode and b.LYPolicyTypeGroup = 'Motor'
        where   c.OwnFaultFlag = 'Y'
        and     cast(c.ClaimDate as timestamp) >= add_months(timestamp '${ads.start_time}', -1) and cast(c.ClaimDate as timestamp) < timestamp '${ads.end_time}'
        group by b.PolicyCode, date_trunc('MONTH', c.ClaimDate)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R1a: Renewal invitation sent by EMAIL


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_motor_renewal_email_offered                                                 */
/*  Description : First 'Renewal Offer - Emailed Document' per renewing motor policy and month (stg.R1_ */
/*              : Motor_Renewals_EmailOffered).                                                         */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_relay_document_event, stg.adsv2_renewal_base                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_motor_renewal_email_offered as
select  d.PolicyCode, min(d.EventDateTime) as OfferedDateTime
from    ${ads.stg}.adsv2_relay_document_event d
join    (select distinct PolicyCode from ${ads.stg}.adsv2_renewal_base where LYPolicyTypeGroup = 'Motor') b
        on b.PolicyCode = d.PolicyCode
where   d.EventDescription like 'Renewal Offer - Emailed Document %'
and     d.PolicyTypeGroup = 'Motor'
and     d.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and d.EventDateTime < timestamp '${ads.end_time}'
group by d.PolicyCode, date_trunc('MONTH', d.EventDateTime)

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r1a  (R1a)                                                              */
/*  Description : Renewals / Spine. Notebook 04 cells 22, 110. Renewal offer emailed: the first email p */
/*              : er policy and month.                                                                  */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyDocumentEvent, PolicyRenewal           */
/*              : Reads: stg.adsv2_motor_renewal_email_offered                                          */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r1a as
select  xxhash64('R1a', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R1a'                                               as EventTypeId,
        'Renewal invitation sent by EMAIL' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  OfferedDateTime                                                 as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.stg}.adsv2_motor_renewal_email_offered
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R1b: Renewal invitation sent by POST


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_motor_renewal_post_offered                                                  */
/*  Description : First 'Renewal Offer - Document Transmitted' per renewing motor policy and month (stg */
/*              : .R1_Motor_Renewals_PostOffered).                                                      */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_relay_document_event, stg.adsv2_renewal_base                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_motor_renewal_post_offered as
select  d.PolicyCode, min(d.EventDateTime) as OfferedDateTime
from    ${ads.stg}.adsv2_relay_document_event d
join    (select distinct PolicyCode from ${ads.stg}.adsv2_renewal_base where LYPolicyTypeGroup = 'Motor') b
        on b.PolicyCode = d.PolicyCode
where   d.EventDescription like 'Renewal Offer - Document Transmitted %'
and     d.PolicyTypeGroup = 'Motor'
and     d.EventDateTime >= timestamp '${ads.start_time}' and d.EventDateTime < timestamp '${ads.end_time}'
group by d.PolicyCode, date_trunc('MONTH', d.EventDateTime)

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r1b  (R1b)                                                              */
/*  Description : Renewals / Spine. Notebook 04 cells 55, 111. Renewal offer posted (document transmitt */
/*              : ed): the first per policy and month.                                                  */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyDocumentEvent, PolicyRenewal           */
/*              : Reads: stg.adsv2_motor_renewal_post_offered                                           */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r1b as
select  xxhash64('R1b', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R1b'                                               as EventTypeId,
        'Renewal invitation sent by POST' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  OfferedDateTime                                                 as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.stg}.adsv2_motor_renewal_post_offered
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R1.B2: Motor Renewal - Alternative offer sent


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r1_b2  (R1.B2)                                                          */
/*  Description : Renewals / Branch. Notebook 04 cell 112. Renewals whose invite premium differs from t */
/*              : he offer (an alternative offer), at the renew date.                                   */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyRenewal                                */
/*              : Reads: ods_v2.PolicyRenewal                                                           */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r1_b2 as
select  xxhash64('R1.B2', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R1.B2'                                             as EventTypeId,
        'Motor Renewal - Alternative offer sent' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(min(LYPolicyRenewDateAdj) as timestamp)                    as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.ods}.PolicyRenewal
        where   LYPolicyTypeGroup = 'Motor'
        and     cast(LYPolicyRenewDateAdj as timestamp) >= add_months(timestamp '${ads.start_time}', -1) and cast(LYPolicyRenewDateAdj as timestamp) < timestamp '${ads.end_time}'
        and     PolicyOfferNum = 1
        and     RenEURPremInvite <> RenEURPremOffer
        group by PolicyCode, date_trunc('MONTH', LYPolicyRenewDateAdj)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R1.E1: Renewal campaigns: CRM


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r1_e1  (R1.E1)                                                          */
/*  Description : Renewals / Chill Effort. Notebook 04 cell 113. Motor renewal CRM campaigns (email / S */
/*              : MS sent) to renewing policies, first per policy and month. Change: read from the Xtre */
/*              : mepush events (04 read ods.scvcampaignhistory, a monthly summary of the same events w */
/*              : ith the same policy code).                                                            */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: MessageEvent, PolicyRenewal                  */
/*              : Reads: ods_v2.MessageEvent, stg.adsv2_renewal_base                                    */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r1_e1 as
select  xxhash64('R1.E1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R1.E1'                                             as EventTypeId,
        'Renewal campaigns: CRM' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(m.EventDateTime)                                            as EventDateTime,
                21                                                              as SourceSystemId,
                m.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.ods}.MessageEvent m
        join    (select distinct PolicyCode from ${ads.stg}.adsv2_renewal_base where LYPolicyTypeGroup = 'Motor') b
                on b.PolicyCode = m.PolicyCode
        where   m.CampaignName like '%Motor%'
        and     m.CampaignName like '%Renewals%'
        and     m.MessageType in ('EMAIL', 'SMS')
        and     m.InteractionType = 'sent'
        and     m.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and m.EventDateTime < add_months(timestamp '${ads.end_time}', 1)
        group by m.PolicyCode, date_trunc('MONTH', m.EventDateTime)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R1.E1.F3: Motor Renewal - Renewal customer in an acquisition campaign audience


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r1_e1_f3  (R1.E1.F3)                                                    */
/*  Description : Renewals / Friction. Notebook 04 cell 114. Renewing policies sent a motor 'TYR' (acqu */
/*              : isition) campaign, first per policy and month.                                        */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: MessageEvent, PolicyRenewal                  */
/*              : Reads: ods_v2.MessageEvent, stg.adsv2_renewal_base                                    */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r1_e1_f3 as
select  xxhash64('R1.E1.F3', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R1.E1.F3'                                          as EventTypeId,
        'Motor Renewal - Renewal customer in an acquisition campaign audience' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(m.EventDateTime)                                            as EventDateTime,
                21                                                              as SourceSystemId,
                m.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.ods}.MessageEvent m
        join    (select distinct PolicyCode from ${ads.stg}.adsv2_renewal_base where LYPolicyTypeGroup = 'Motor') b
                on b.PolicyCode = m.PolicyCode
        where   m.CampaignName like '%Motor%'
        and     m.CampaignName like '%TYR%'
        and     m.MessageType in ('EMAIL', 'SMS')
        and     m.InteractionType = 'sent'
        and     m.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and m.EventDateTime < timestamp '${ads.end_time}'
        group by m.PolicyCode, date_trunc('MONTH', m.EventDateTime)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R2a: Motor Renewal - Customer logs in to portal


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r2a  (R2a)                                                              */
/*  Description : Renewals / Spine. Notebook 04 cell 115. Portal logins by the policy's client from 60  */
/*              : days before to 40 days after the renew date.                                          */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: RenewalPortalEvent, PolicyRenewal            */
/*              : Reads: stg.adsv2_renewal_base, ods_v2.RenewalPortalEvent                              */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r2a as
select  xxhash64('R2a', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R2a'                                               as EventTypeId,
        'Motor Renewal - Customer logs in to portal' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(e.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.ods}.RenewalPortalEvent e on e.PortfolioCode = left(b.PolicyCode, 6)
        where   b.LYPolicyTypeGroup = 'Motor'
        and     e.EventType = 'LoginSuccess'
        and     e.EventDateTime > date_sub(b.RenewalDate, 60)
        and     e.EventDateTime < date_add(b.RenewalDate, 40)
        group by b.PolicyCode, date_trunc('MONTH', e.EventDateTime)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R2b: Motor Renewal - Customer phones call cente


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_motor_renewal_call                                                          */
/*  Description : Genesys sessions on the motor renewals queue in the window, per conversation (stg.R0_ */
/*              : genesys_derived_data / R3bF1 summary).                                                */
/*              :                                                                                       */
/*              : Reads: ods_v2.CallSession                                                             */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_motor_renewal_call as
select  ConversationId,
        min(ConversationStartTime)                                      as ConversationStartTime,
        sum(Abandoned)                                                  as Abandoned,
        sum(AcdWaitDurationMs)                                          as CallWaitTime
from    ${ads.ods}.CallSession
where   QueueName = 'INBOUND_Car_Renewals'
and     ConversationStartTime >= timestamp '${ads.start_time}' and ConversationStartTime < timestamp '${ads.end_time}'
group by ConversationId

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r2b  (R2b)                                                              */
/*  Description : Renewals / Spine. Notebook 04 cells 26, 116. Conversations on the INBOUND_Car_Renewal */
/*              : s queue, at their start. Change: source system 20 Genesys (04: 2).                    */
/*              :                                                                                       */
/*              : Grain: Call   Product: Motor   Source: CallSession                                    */
/*              : Reads: stg.adsv2_motor_renewal_call                                                   */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r2b as
select  xxhash64('R2b', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R2b'                                               as EventTypeId,
        'Motor Renewal - Customer phones call cente' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        cast(null as string)                                         as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  ConversationStartTime                                           as EventDateTime,
                20                                                              as SourceSystemId,
                ConversationId                                                  as EventSourceReference
        from    ${ads.stg}.adsv2_motor_renewal_call
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R2.F1: Failed login


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r2_f1  (R2.F1)                                                          */
/*  Description : Renewals / Friction. Notebook 04 cell 117. Failed portal logins in the renewal period */
/*              :  (as R2a).                                                                            */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: RenewalPortalEvent, PolicyRenewal            */
/*              : Reads: stg.adsv2_renewal_base, ods_v2.RenewalPortalEvent                              */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r2_f1 as
select  xxhash64('R2.F1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R2.F1'                                             as EventTypeId,
        'Failed login' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(e.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.ods}.RenewalPortalEvent e on e.PortfolioCode = left(b.PolicyCode, 6)
        where   b.LYPolicyTypeGroup = 'Motor'
        and     e.EventType = 'LoginFailed'
        and     e.EventDateTime > date_sub(b.RenewalDate, 60)
        and     e.EventDateTime < date_add(b.RenewalDate, 40)
        group by b.PolicyCode, date_trunc('MONTH', e.EventDateTime)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R2.B1.F1: The customer shops the renewal at all


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_motor_renewal_mfq                                                           */
/*  Description : MFQ re-quotes of each renewing motor cycle, due - 60 to due + 30 days (H8; stg.Renewa */
/*              : lsDoingMFQ).                                                                          */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_renewal_scv_quotes                                                   */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_motor_renewal_mfq as
select  q.PolicyCode, q.RenewalDate, q.QuoteReference as GuidKey, q.QuoteId, q.QuoteDateTime
from    ${ads.stg}.adsv2_renewal_scv_quotes q
where   q.LYPolicyTypeGroup = 'Motor'
and     q.QuoteSourceSystemId = 2

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r2_b1_f1  (R2.B1.F1)                                                    */
/*  Description : Renewals / Friction. Notebook 04 cells 24, 118, corrected (H8, D-E1.27): MFQ re-quote */
/*              : s of the cycle (same customer, due - 60 to due + 30) with no successful MFQ payment o */
/*              : n that cycle's re-quotes; one event per re-quote, at its start.                       */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: Quote, QuoteMotorPayment, scveadmlinkfinal   */
/*              : Reads: stg.adsv2_motor_renewal_mfq, ods_v2.QuoteMotorPayment                          */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r2_b1_f1 as
select  xxhash64('R2.B1.F1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R2.B1.F1'                                          as EventTypeId,
        'The customer shops the renewal at all' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(a.QuoteDateTime)                                            as EventDateTime,
                2                                                               as SourceSystemId,
                a.PolicyCode                                                    as SourcePolicyReference,
                a.GuidKey                                                       as SourceQuoteReference,
                max(a.QuoteId)                                                  as QuoteId
        from    ${ads.stg}.adsv2_motor_renewal_mfq a
        left anti join (
                select  r.PolicyCode, r.RenewalDate
                from    ${ads.stg}.adsv2_motor_renewal_mfq r
                join    ${ads.ods}.QuoteMotorPayment p  on p.GuidKey = r.GuidKey
                where   p.PaymentStatus in (0, 5)
                ) b     on b.PolicyCode = a.PolicyCode and b.RenewalDate = a.RenewalDate
        group by a.PolicyCode, a.RenewalDate, a.GuidKey
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R2.E1: Motor Renewal - In-flight price improvement: promo code, or the agent reduces the fee


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r2_e1  (R2.E1)                                                          */
/*  Description : Renewals / Chill EFfort. Notebook 04 cell 119. Renewals whose fee was changed from th */
/*              : e offered fee (promo code or agent reduction), on the new policy at its sale date. Ch */
/*              : ange: renewals with no sale date are left out (04 inserted them without a time).      */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyRenewal                                */
/*              : Reads: stg.adsv2_renewal_base, ods_v2.PolicyRenewal                                   */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r2_e1 as
select  xxhash64('R2.E1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R2.E1'                                             as EventTypeId,
        'Motor Renewal - In-flight price improvement: promo code, or the agent reduces the fee' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(min(b.TYReportingSaleDate) as timestamp)                   as EventDateTime,
                1                                                               as SourceSystemId,
                b.TYPolicyCode                                                  as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.ods}.PolicyRenewal r      on r.PolicyCode = b.PolicyCode
        where   b.LYPolicyTypeGroup = 'Motor'
        and     r.RenewalMonth < date '${ads.end_time}'
        and     r.RenEURFee <> r.TYEURFees
        group by b.TYPolicyCode, date_trunc('MONTH', b.TYReportingSaleDate)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R3a: Customer renews ONLINE


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_motor_renewal_online                                                        */
/*  Description : Motor renewals paid online (stg.MotorRenewalsOnline); SalesSource 1 = portal, same po */
/*              : licy, 2 = portal, same client, 3 = MFQ.                                               */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_renewal_base, ods_v2.RenewalPortalEvent, stg.adsv2_motor_renewal_mfq */
/*              : , ods_v2.QuoteMotorPayment                                                            */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_motor_renewal_online as
select  distinct b.PolicyCode, 1 as SalesSource, cast(date_sub(b.RenewalDate, 60) as timestamp) as RenewalStartDate
from    ${ads.stg}.adsv2_renewal_base b
join    ${ads.ods}.RenewalPortalEvent e on e.PolicyCode = b.PolicyCode
where   b.LYPolicyTypeGroup = 'Motor'
and     e.EventType = 'PaymentSuccess'
and     e.EventDateTime > date_sub(b.RenewalDate, 60)
and     e.EventDateTime < date_add(b.RenewalDate, 40)
union
select  distinct e.PolicyCode, 2, cast(date_sub(b.RenewalDate, 60) as timestamp)
from    ${ads.stg}.adsv2_renewal_base b
join    ${ads.ods}.RenewalPortalEvent e on left(e.PolicyCode, 6) = left(b.PolicyCode, 6) and e.PolicyCode <> b.PolicyCode
where   b.LYPolicyTypeGroup = 'Motor'
and     e.EventType = 'PaymentSuccess'
and     e.EventDateTime > date_sub(b.RenewalDate, 60)
and     e.EventDateTime < date_add(b.RenewalDate, 40)
union
select  distinct r.PolicyCode, 3, p.DateUpdated
from    ${ads.stg}.adsv2_motor_renewal_mfq r
join    ${ads.ods}.QuoteMotorPayment p  on p.GuidKey = r.GuidKey
where   p.PaymentStatus in (0, 5)

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r3a  (R3a)                                                              */
/*  Description : Renewals / Spine. Notebook 04 cells 25, 120. Renewed online: a portal payment for the */
/*              :  policy (1) or another policy of the client (2) in the renewal period, at the renewal */
/*              :  start (renew date - 60), or a successful MFQ payment on the customer's renewal-time  */
/*              : MFQ quote (3), at the payment. Latest per policy and month.                           */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: RenewalPortalEvent, QuoteMotorPayment, Polic */
/*              : yRenewal                                                                              */
/*              : Reads: stg.adsv2_motor_renewal_online                                                 */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r3a as
select  xxhash64('R3a', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R3a'                                               as EventTypeId,
        'Customer renews ONLINE' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(RenewalStartDate)                                           as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.stg}.adsv2_motor_renewal_online
        group by PolicyCode, date_trunc('MONTH', RenewalStartDate)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R3b: Customer renews on an INBOUND call


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r3b  (R3b)                                                              */
/*  Description : Renewals / Spine. Notebook 04 cell 121. Motor (M1) renewal transactions posted in the */
/*              :  window, latest per policy and month.                                                 */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyTransaction                            */
/*              : Reads: ods_v2.PolicyTransaction                                                       */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r3b as
select  xxhash64('R3b', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R3b'                                               as EventTypeId,
        'Customer renews on an INBOUND call' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(max(PostingDate) as timestamp)                             as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.ods}.PolicyTransaction
        where   PremiumTypeGroup = 'Renewals'
        and     PolicyType = 'M1'
        and     cast(PostingDate as timestamp) >= timestamp '${ads.start_time}' and cast(PostingDate as timestamp) < timestamp '${ads.end_time}'
        group by PolicyCode, date_trunc('MONTH', PostingDate)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R3c: Customer renews on an OUTBOUND call


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r3c  (R3c)                                                              */
/*  Description : Renewals / Spine. Notebook 04 cell 122. Retained first-offer motor renewals on channe */
/*              : l 'a) IB', on the new policy at the renew date.                                       */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyRenewal                                */
/*              : Reads: ods_v2.PolicyRenewal                                                           */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r3c as
select  xxhash64('R3c', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R3c'                                               as EventTypeId,
        'Customer renews on an OUTBOUND call' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(max(LYPolicyRenewDateAdj) as timestamp)                    as EventDateTime,
                1                                                               as SourceSystemId,
                TYPolicyCode                                                    as SourcePolicyReference
        from    ${ads.ods}.PolicyRenewal
        where   RenewalMonth < date '${ads.end_time}'
        and     Channel = 'a) IB'
        and     PolicyRetNum = 1
        and     PolicyOfferNum = 1
        and     LYPolicyTypeGroup = 'Motor'
        group by TYPolicyCode, date_trunc('MONTH', LYPolicyRenewDateAdj)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R3d: Motor Renewal - Customer renews on CHAT


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r3d  (R3d)                                                              */
/*  Description : Renewals / Spine. Notebook 04 cell 123. As R3c on a web channel (Channel like '%eb %' */
/*              : ).                                                                                    */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyRenewal                                */
/*              : Reads: ods_v2.PolicyRenewal                                                           */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r3d as
select  xxhash64('R3d', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R3d'                                               as EventTypeId,
        'Motor Renewal - Customer renews on CHAT' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(max(LYPolicyRenewDateAdj) as timestamp)                    as EventDateTime,
                1                                                               as SourceSystemId,
                TYPolicyCode                                                    as SourcePolicyReference
        from    ${ads.ods}.PolicyRenewal
        where   RenewalMonth < date '${ads.end_time}'
        and     Channel like '%eb %'
        and     PolicyRetNum = 1
        and     PolicyOfferNum = 1
        and     LYPolicyTypeGroup = 'Motor'
        group by TYPolicyCode, date_trunc('MONTH', LYPolicyRenewDateAdj)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R3b.F1: Call wait time-  or fails to make contact


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r3b_f1  (R3b.F1)                                                        */
/*  Description : Renewals / Friction. Notebook 04 cells 27, 124. Renewal-queue conversations that wait */
/*              : ed more than 600 seconds or were abandoned (D-E1.25). Change: source system 20 Genesy */
/*              : s.                                                                                    */
/*              :                                                                                       */
/*              : Grain: Call   Product: Motor   Source: CallSession                                    */
/*              : Reads: stg.adsv2_motor_renewal_call                                                   */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r3b_f1 as
select  xxhash64('R3b.F1', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R3b.F1'                                            as EventTypeId,
        'Call wait time-  or fails to make contact' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        cast(null as string)                                         as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  ConversationStartTime                                           as EventDateTime,
                20                                                              as SourceSystemId,
                ConversationId                                                  as EventSourceReference
        from    ${ads.stg}.adsv2_motor_renewal_call
        where   CallWaitTime > 600000
        or      Abandoned > 0
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R3.O1: Policy lapsed


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r3_o1  (R3.O1)                                                          */
/*  Description : Renewals / Outcome. Notebook 04, corrected (H10, D-E1.21): the policy lapsed in this  */
/*              : cycle = lost (not renewed within the window), from the cycle's own result, at due + 3 */
/*              : 0 days. 04 wrote the policy's current status L at every renewal date.                 */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyRenewal, PolicySale, RenewalPortalEven */
/*              : t                                                                                     */
/*              : Reads: stg.adsv2_renewal_base                                                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r3_o1 as
select  xxhash64('R3.O1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R3.O1'                                             as EventTypeId,
        'Policy lapsed' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(date_add(RenewalDate, 30) as timestamp)                       as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base
        where   LYPolicyTypeGroup = 'Motor'
        and     Lost = 1
        and     OutcomeFinal = 1
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R4: Payment taken


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r4  (R4)                                                                */
/*  Description : Renewals / Spine. Notebook 04 cell 126. Retained motor renewals (payment taken), on t */
/*              : he new policy at the renew date.                                                      */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyRenewal                                */
/*              : Reads: stg.adsv2_renewal_base                                                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r4 as
select  xxhash64('R4', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R4'                                                as EventTypeId,
        'Payment taken' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(max(RenewalDate) as timestamp)                             as EventDateTime,
                1                                                               as SourceSystemId,
                TYPolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base
        where   LYPolicyTypeGroup = 'Motor'
        and     PolicyRetNum = 1
        group by TYPolicyCode, date_trunc('MONTH', RenewalDate)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R4.F1: Payment failed


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r4_f1  (R4.F1)                                                          */
/*  Description : Renewals / Friction. Notebook 04 cell 127. Failed portal payments by the client in th */
/*              : e renewal period, at the renewal start (as 04).                                       */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: RenewalPortalEvent, PolicyRenewal            */
/*              : Reads: stg.adsv2_renewal_base, ods_v2.RenewalPortalEvent                              */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r4_f1 as
select  xxhash64('R4.F1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R4.F1'                                             as EventTypeId,
        'Payment failed' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(max(date_sub(b.RenewalDate, 60)) as timestamp)             as EventDateTime,
                1                                                               as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.ods}.RenewalPortalEvent e on e.PortfolioCode = left(b.PolicyCode, 6)
        where   b.LYPolicyTypeGroup = 'Motor'
        and     e.EventType = 'PaymentFailed'
        and     e.EventDateTime > date_sub(b.RenewalDate, 60)
        and     e.EventDateTime < date_add(b.RenewalDate, 40)
        group by b.PolicyCode, date_trunc('MONTH', date_sub(b.RenewalDate, 60))
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R4a: Pay in full


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r4a  (R4a)                                                              */
/*  Description : Renewals / Spine. Notebook 04 cell 128. Portal payments in full in the renewal period */
/*              : , at the renewal start.                                                               */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: RenewalPortalEvent, PolicyRenewal            */
/*              : Reads: stg.adsv2_renewal_base, ods_v2.RenewalPortalEvent                              */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r4a as
select  xxhash64('R4a', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R4a'                                               as EventTypeId,
        'Pay in full' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(max(date_sub(b.RenewalDate, 60)) as timestamp)             as EventDateTime,
                1                                                               as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.ods}.RenewalPortalEvent e on e.PortfolioCode = left(b.PolicyCode, 6)
        where   b.LYPolicyTypeGroup = 'Motor'
        and     e.EventType = 'PaymentSuccess'
        and     e.PaymentType = 'Full'
        and     e.EventDateTime > date_sub(b.RenewalDate, 60)
        and     e.EventDateTime < date_add(b.RenewalDate, 40)
        group by b.PolicyCode, date_trunc('MONTH', date_sub(b.RenewalDate, 60))
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R4c: New monthly agreement


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r4c  (R4c)                                                              */
/*  Description : Renewals / Spine. Notebook 04 cell 129. Portal payments by instalments (new monthly a */
/*              : greement) in the renewal period.                                                      */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: RenewalPortalEvent, PolicyRenewal            */
/*              : Reads: stg.adsv2_renewal_base, ods_v2.RenewalPortalEvent                              */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r4c as
select  xxhash64('R4c', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R4c'                                               as EventTypeId,
        'New monthly agreement' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(max(date_sub(b.RenewalDate, 60)) as timestamp)             as EventDateTime,
                1                                                               as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.ods}.RenewalPortalEvent e on e.PortfolioCode = left(b.PolicyCode, 6)
        where   b.LYPolicyTypeGroup = 'Motor'
        and     e.EventType = 'PaymentSuccess'
        and     e.PaymentType = 'Instalments'
        and     e.EventDateTime > date_sub(b.RenewalDate, 60)
        and     e.EventDateTime < date_add(b.RenewalDate, 40)
        group by b.PolicyCode, date_trunc('MONTH', date_sub(b.RenewalDate, 60))
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R5: Renewal processed-  policy continued


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r5  (R5)                                                                */
/*  Description : Renewals / Spine. Notebook 04 cell 130. A suitability statement on the new (this-year */
/*              : ) policy: renewal processed.                                                          */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyDocumentEvent, PolicyRenewal           */
/*              : Reads: stg.adsv2_renewal_base, stg.adsv2_relay_document_event                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r5 as
select  xxhash64('R5', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R5'                                                as EventTypeId,
        'Renewal processed-  policy continued' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = b.TYPolicyCode
        where   b.LYPolicyTypeGroup = 'Motor'
        and     d.PolicyTypeGroup = 'Motor'
        and     d.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and d.EventDateTime < timestamp '${ads.end_time}'
        and     d.EventDescription in ('Renewal Confirmation - Emailed Document - Suitability Statement',
                                       'Renewal Transfer - Emailed Document - Motor Suitability Statement',
                                       'New Business - Emailed Document - Suitability Statement',
                                       'Renewal Confirmation - Document Transmitted - Suitability Statement',
                                       'Renewal Transfer - Document Transmitted - Motor Suitability Statement',
                                       'New Business - Document Transmitted - Suitability Statement',
                                       'New Business - Document Transmitted - Motor Suitability Statement')
        group by b.PolicyCode, date_trunc('MONTH', d.EventDateTime)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R6: Cert and disc sent


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r6  (R6)                                                                */
/*  Description : Renewals / Spine. Notebook 04 cell 131. A certificate (or cert issue letter) on the n */
/*              : ew policy.                                                                            */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyDocumentEvent, PolicyRenewal           */
/*              : Reads: stg.adsv2_renewal_base, stg.adsv2_relay_document_event                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r6 as
select  xxhash64('R6', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R6'                                                as EventTypeId,
        'Cert and disc sent' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = b.TYPolicyCode
        where   b.LYPolicyTypeGroup = 'Motor'
        and     d.PolicyTypeGroup = 'Motor'
        and     d.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and d.EventDateTime < timestamp '${ads.end_time}'
        and     d.EventDescription in ('Renewal Confirmation - Document Transmitted - RNL Cert Issue Letter',
                                       'Renewal Confirmation - Document Transmitted - Certificate',
                                       'Renewal Transfer - Document Transmitted - RNL Cert Issue Letter',
                                       'Renewal Transfer - Document Transmitted - Certificate',
                                       'New Business - Document Transmitted - NB Cert Issue Letter',
                                       'New Business - Document Transmitted - Certificate',
                                       'Renewal Confirmation - Emailed Document - Certificate',
                                       'Renewal Transfer - Emailed Document - Certificate',
                                       'Renewal Transfer - Printed Document - RNL Cert Issue Letter')
        group by b.PolicyCode, date_trunc('MONTH', d.EventDateTime)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R2.B1.1: MFQ quote started from the renewal price


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_motor_renewal_offer_quotes                                                  */
/*  Description : Each cycle's MFQ re-quotes with the renewal offer and how the cycle renewed (H8; stg. */
/*              : R2B1_1_Base).                                                                         */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_motor_renewal_mfq, stg.adsv2_renewal_base                            */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_motor_renewal_offer_quotes as
select  q.PolicyCode, q.RenewalDate, q.GuidKey, q.QuoteId, q.QuoteDateTime,
        b.RenEURPremOffer, b.TYEURGrossPremium, b.Renewed, b.RenewedSameCode, b.OutcomeFinal
from    ${ads.stg}.adsv2_motor_renewal_mfq q
join    ${ads.stg}.adsv2_renewal_base b   on b.PolicyCode = q.PolicyCode and b.RenewalDate = q.RenewalDate

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r2_b1_1  (R2.B1.1)                                                      */
/*  Description : Renewals / Branch. Notebook 04 cells 28, 132, corrected (H8): an MFQ re-quote of the  */
/*              : cycle (due - 60 to due + 30), one event per re-quote at its start. 04 took quotes wit */
/*              : hin two months after the email offer.                                                 */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: Quote, QuoteMotor, QuoteMotorSummary, Policy */
/*              : Renewal, scveadmlinkfinal                                                             */
/*              : Reads: stg.adsv2_motor_renewal_offer_quotes                                           */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r2_b1_1 as
select  xxhash64('R2.B1.1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R2.B1.1'                                           as EventTypeId,
        'MFQ quote started from the renewal price' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(q.QuoteDateTime)                                          as EventDateTime,
                2                                                               as SourceSystemId,
                q.PolicyCode                                                    as SourcePolicyReference,
                q.GuidKey                                                    as SourceQuoteReference,
                max(q.QuoteId)                                                  as QuoteId
        from    ${ads.stg}.adsv2_motor_renewal_offer_quotes q
        group by q.PolicyCode, q.RenewalDate, q.GuidKey
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R2.B1.2: Details carried over or amended


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r2_b1_2  (R2.B1.2)                                                      */
/*  Description : Renewals / Branch. Notebook 04 cell 133. As R2.B1.1 (the re-quote exists in Quote), o */
/*              : ne event per re-quote.                                                                */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: Quote, QuoteMotor, QuoteMotorSummary, Policy */
/*              : Renewal, scveadmlinkfinal                                                             */
/*              : Reads: stg.adsv2_motor_renewal_offer_quotes, ods_v2.Quote                             */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r2_b1_2 as
select  xxhash64('R2.B1.2', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R2.B1.2'                                           as EventTypeId,
        'Details carried over or amended' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(q.QuoteDateTime)                                          as EventDateTime,
                2                                                               as SourceSystemId,
                q.PolicyCode                                                    as SourcePolicyReference,
                q.GuidKey                                                    as SourceQuoteReference,
                max(q.QuoteId)                                                  as QuoteId
        from    ${ads.stg}.adsv2_motor_renewal_offer_quotes q
        left semi join ${ads.ods}.Quote qq on qq.SourceSystemId = 2 and qq.SourceSystemQuoteReference = q.GuidKey
        group by q.PolicyCode, q.RenewalDate, q.GuidKey
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R2.B1.3: New price returned


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_motor_renewal_priced_quotes                                                 */
/*  Description : The cycle's re-quotes with at least one insurer price, with the cheapest comprehensiv */
/*              : e premium (stg.R2B1_3_base / R2B1_4_F1_MFQPrices).                                    */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_motor_renewal_offer_quotes, ods_v2.QuoteMotor, ods_v2.QuoteMotorSumm */
/*              : ary                                                                                   */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_motor_renewal_priced_quotes as
select  o.*, s.Rank1PremComp
from    ${ads.stg}.adsv2_motor_renewal_offer_quotes o
join    ${ads.ods}.QuoteMotor m             on m.GuidKey = o.GuidKey
left join ${ads.ods}.QuoteMotorSummary s    on s.GuidKey = o.GuidKey
where   m.InsurerCount > 0

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r2_b1_3  (R2.B1.3)                                                      */
/*  Description : Renewals / Branch. Notebook 04 cells 29, 134. A re-quote of the cycle with at least o */
/*              : ne insurer price (QuoteMotor.InsurerCount > 0).                                       */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: Quote, QuoteMotor, QuoteMotorSummary, Policy */
/*              : Renewal, scveadmlinkfinal                                                             */
/*              : Reads: stg.adsv2_motor_renewal_priced_quotes                                          */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r2_b1_3 as
select  xxhash64('R2.B1.3', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R2.B1.3'                                           as EventTypeId,
        'New price returned' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(q.QuoteDateTime)                                          as EventDateTime,
                2                                                               as SourceSystemId,
                q.PolicyCode                                                    as SourcePolicyReference,
                q.GuidKey                                                    as SourceQuoteReference,
                max(q.QuoteId)                                                  as QuoteId
        from    ${ads.stg}.adsv2_motor_renewal_priced_quotes q
        group by q.PolicyCode, q.RenewalDate, q.GuidKey
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R2.B1.3.F1: No price returned-  or the quote is abandoned


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r2_b1_3_f1  (R2.B1.3.F1)                                                */
/*  Description : Renewals / Friction. Notebook 04 cell 135, corrected (H8): re-quotes of a cycle in wh */
/*              : ich no re-quote got a price; one event per re-quote. 04 excluded the policy if any qu */
/*              : ote in any year got a price.                                                          */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: Quote, QuoteMotor, scveadmlinkfinal          */
/*              : Reads: stg.adsv2_motor_renewal_offer_quotes, stg.adsv2_motor_renewal_priced_quotes    */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r2_b1_3_f1 as
select  xxhash64('R2.B1.3.F1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R2.B1.3.F1'                                        as EventTypeId,
        'No price returned-  or the quote is abandoned' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(q.QuoteDateTime)                                            as EventDateTime,
                2                                                               as SourceSystemId,
                q.PolicyCode                                                    as SourcePolicyReference,
                q.GuidKey                                                       as SourceQuoteReference,
                max(q.QuoteId)                                                  as QuoteId
        from    ${ads.stg}.adsv2_motor_renewal_offer_quotes q
        left anti join (select distinct PolicyCode, RenewalDate from ${ads.stg}.adsv2_motor_renewal_priced_quotes) p
                on p.PolicyCode = q.PolicyCode and p.RenewalDate = q.RenewalDate
        group by q.PolicyCode, q.RenewalDate, q.GuidKey
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R2.B1.4.F1: New price is worse than the renewal offer


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r2_b1_4_f1  (R2.B1.4.F1)                                                */
/*  Description : Renewals / Friction. Notebook 04 cells 30, 136. The cheapest comprehensive MFQ price  */
/*              : is worse than the renewal offer (H8: compared with the offer; the price is the latest */
/*              :  month-end QuoteMotorSummary row).                                                    */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: Quote, QuoteMotor, QuoteMotorSummary, Policy */
/*              : Renewal, scveadmlinkfinal                                                             */
/*              : Reads: stg.adsv2_motor_renewal_priced_quotes                                          */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r2_b1_4_f1 as
select  xxhash64('R2.B1.4.F1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R2.B1.4.F1'                                        as EventTypeId,
        'New price is worse than the renewal offer' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(q.QuoteDateTime)                                          as EventDateTime,
                2                                                               as SourceSystemId,
                q.PolicyCode                                                    as SourcePolicyReference,
                q.GuidKey                                                    as SourceQuoteReference,
                max(q.QuoteId)                                                  as QuoteId
        from    ${ads.stg}.adsv2_motor_renewal_priced_quotes q
        where   q.Rank1PremComp > q.RenEURPremOffer
        and     q.RenEURPremOffer > 0
        group by q.PolicyCode, q.RenewalDate, q.GuidKey
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R2.B1.O1: Renews on the renewal offer


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r2_b1_o1  (R2.B1.O1)                                                    */
/*  Description : Renewals / Outcome. Notebook 04 cell 137, corrected (H8, D-E1.27): a priced re-quote, */
/*              :  and the cycle renewed on the renewal offer (renewed on the same policy code).        */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: Quote, QuoteMotor, QuoteMotorSummary, Policy */
/*              : Renewal, scveadmlinkfinal                                                             */
/*              : Reads: stg.adsv2_motor_renewal_priced_quotes                                          */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r2_b1_o1 as
select  xxhash64('R2.B1.O1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R2.B1.O1'                                          as EventTypeId,
        'Renews on the renewal offer' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(q.QuoteDateTime)                                          as EventDateTime,
                2                                                               as SourceSystemId,
                q.PolicyCode                                                    as SourcePolicyReference,
                q.GuidKey                                                    as SourceQuoteReference,
                max(q.QuoteId)                                                  as QuoteId
        from    ${ads.stg}.adsv2_motor_renewal_priced_quotes q
        where   q.Renewed = 1
        and     q.RenewedSameCode = 1
        group by q.PolicyCode, q.RenewalDate, q.GuidKey
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R2.B1.O2: Renews on a new MFQ price


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r2_b1_o2  (R2.B1.O2)                                                    */
/*  Description : Renewals / Outcome. Notebook 04 cell 138, corrected (H8, D-E1.27): a priced re-quote, */
/*              :  and the cycle renewed on a new price (renewed onto a new policy code, by the finance */
/*              :  or the +/-30-day match). 04 tested 'offer above the premium paid'.                   */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: Quote, QuoteMotor, QuoteMotorSummary, Policy */
/*              : Renewal, scveadmlinkfinal                                                             */
/*              : Reads: stg.adsv2_motor_renewal_priced_quotes                                          */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r2_b1_o2 as
select  xxhash64('R2.B1.O2', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R2.B1.O2'                                          as EventTypeId,
        'Renews on a new MFQ price' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(q.QuoteDateTime)                                          as EventDateTime,
                2                                                               as SourceSystemId,
                q.PolicyCode                                                    as SourcePolicyReference,
                q.GuidKey                                                    as SourceQuoteReference,
                max(q.QuoteId)                                                  as QuoteId
        from    ${ads.stg}.adsv2_motor_renewal_priced_quotes q
        where   q.Renewed = 1
        and     q.RenewedSameCode = 0
        group by q.PolicyCode, q.RenewalDate, q.GuidKey
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R2.B1.O3: Buys elsewhere-  no renewal


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r2_b1_o3  (R2.B1.O3)                                                    */
/*  Description : Renewals / Outcome. Notebook 04 cell 139, corrected (H8, D-E1.27): a priced re-quote, */
/*              :  and the cycle did not renew (buys elsewhere), once the outcome is final.             */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: Quote, QuoteMotor, QuoteMotorSummary, Policy */
/*              : Renewal, scveadmlinkfinal                                                             */
/*              : Reads: stg.adsv2_motor_renewal_priced_quotes                                          */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r2_b1_o3 as
select  xxhash64('R2.B1.O3', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R2.B1.O3'                                          as EventTypeId,
        'Buys elsewhere-  no renewal' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(q.QuoteDateTime)                                          as EventDateTime,
                2                                                               as SourceSystemId,
                q.PolicyCode                                                    as SourcePolicyReference,
                q.GuidKey                                                    as SourceQuoteReference,
                max(q.QuoteId)                                                  as QuoteId
        from    ${ads.stg}.adsv2_motor_renewal_priced_quotes q
        where   q.Renewed = 0
        and     q.OutcomeFinal = 1
        group by q.PolicyCode, q.RenewalDate, q.GuidKey
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R3.B1: This year lapsed


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r3_b1  (R3.B1)                                                          */
/*  Description : Renewals / Branch. Notebook 04, corrected. Lapsed = no renewal by the due date (D-E1. */
/*              : 21), from the cycle's resolved renewal (finance retained, else the +/-30-day match; D */
/*              : -E1.19). One event per cycle, at the due date, once the due date has passed. 04 count */
/*              : ed renewals not yet due as lapsed, and (Home) left out lapses with no renewal at all. */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyRenewal, PolicySale, RenewalPortalEven */
/*              : t                                                                                     */
/*              : Reads: stg.adsv2_renewal_base                                                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r3_b1 as
select  xxhash64('R3.B1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R3.B1'                                             as EventTypeId,
        'This year lapsed' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(RenewalDate as timestamp)                                     as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base
        where   LYPolicyTypeGroup = 'Motor'
        and     Lapsed = 1
        and     RenewalDate < date '${ads.end_time}'
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R3.B1.1: Renewal date passes with no renewal


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r3_b1_1  (R3.B1.1)                                                      */
/*  Description : Renewals / Branch. Notebook 04, corrected. Lapsed = no renewal by the due date (D-E1. */
/*              : 21), from the cycle's resolved renewal (finance retained, else the +/-30-day match; D */
/*              : -E1.19). One event per cycle, at the due date, once the due date has passed. 04 count */
/*              : ed renewals not yet due as lapsed, and (Home) left out lapses with no renewal at all. */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyRenewal, PolicySale, RenewalPortalEven */
/*              : t                                                                                     */
/*              : Reads: stg.adsv2_renewal_base                                                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r3_b1_1 as
select  xxhash64('R3.B1.1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R3.B1.1'                                           as EventTypeId,
        'Renewal date passes with no renewal' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(RenewalDate as timestamp)                                     as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base
        where   LYPolicyTypeGroup = 'Motor'
        and     Lapsed = 1
        and     RenewalDate < date '${ads.end_time}'
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R3.B1.1.F1: Motor Renewal - CRM bounces


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r3_b1_1_f1  (R3.B1.1.F1)                                                */
/*  Description : Renewals / Friction. Notebook 04 cell 142. Bounced motor 'Lapsed' campaign messages t */
/*              : o renewing policies, first per policy and month.                                      */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: MessageEvent, PolicyRenewal                  */
/*              : Reads: ods_v2.MessageEvent, stg.adsv2_renewal_base                                    */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r3_b1_1_f1 as
select  xxhash64('R3.B1.1.F1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R3.B1.1.F1'                                        as EventTypeId,
        'Motor Renewal - CRM bounces' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(m.EventDateTime)                                            as EventDateTime,
                21                                                              as SourceSystemId,
                m.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.ods}.MessageEvent m
        join    (select distinct PolicyCode from ${ads.stg}.adsv2_renewal_base where LYPolicyTypeGroup = 'Motor') b
                on b.PolicyCode = m.PolicyCode
        where   m.CampaignName like '%Motor%'
        and     m.CampaignName like '%Lapsed%'
        and     m.MessageType in ('EMAIL', 'SMS')
        and     m.InteractionType = 'bounce'
        and     m.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and m.EventDateTime < timestamp '${ads.end_time}'
        group by m.PolicyCode, date_trunc('MONTH', m.EventDateTime)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R3.B1.2: Lapse CRM sequence runs


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r3_b1_2  (R3.B1.2)                                                      */
/*  Description : Renewals / Branch. Notebook 04 cell 143. Motor 'Lapsed' campaign messages sent to ren */
/*              : ewing policies (the lapse sequence runs).                                             */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: MessageEvent, PolicyRenewal                  */
/*              : Reads: ods_v2.MessageEvent, stg.adsv2_renewal_base                                    */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r3_b1_2 as
select  xxhash64('R3.B1.2', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R3.B1.2'                                           as EventTypeId,
        'Lapse CRM sequence runs' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(m.EventDateTime)                                            as EventDateTime,
                21                                                              as SourceSystemId,
                m.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.ods}.MessageEvent m
        join    (select distinct PolicyCode from ${ads.stg}.adsv2_renewal_base where LYPolicyTypeGroup = 'Motor') b
                on b.PolicyCode = m.PolicyCode
        where   m.CampaignName like '%Motor%'
        and     m.CampaignName like '%Lapsed%'
        and     m.MessageType in ('EMAIL', 'SMS')
        and     m.InteractionType = 'sent'
        and     m.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and m.EventDateTime < timestamp '${ads.end_time}'
        group by m.PolicyCode, date_trunc('MONTH', m.EventDateTime)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R3.B1.2.F1: Unreachable on the lapse sequence


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r3_b1_2_f1  (R3.B1.2.F1)                                                */
/*  Description : Renewals / Friction. Notebook 04 cell 144, corrected (Vijay, 2026-10-19: the original */
/*              :  selected the policies it then excluded, so it never returned rows). Renewing motor p */
/*              : olicies sent a 'Motor - This Year Lapsed' email that was never opened (no open or cli */
/*              : ck) or that bounced; one event per policy and month, at the last send.                */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: MessageEvent, PolicyRenewal                  */
/*              : Reads: ods_v2.MessageEvent, stg.adsv2_renewal_base                                    */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r3_b1_2_f1 as
select  xxhash64('R3.B1.2.F1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R3.B1.2.F1'                                        as EventTypeId,
        'Unreachable on the lapse sequence' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(case when m.InteractionType = 'sent' then m.EventDateTime end) as EventDateTime,
                21                                                              as SourceSystemId,
                m.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.ods}.MessageEvent m
        join    (select distinct PolicyCode from ${ads.stg}.adsv2_renewal_base where LYPolicyTypeGroup = 'Motor') b
                on b.PolicyCode = m.PolicyCode
        where   m.CampaignName like 'Motor - This Year Lapsed%'
        and     m.MessageType = 'EMAIL'
        and     m.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and m.EventDateTime < timestamp '${ads.end_time}'
        group by m.PolicyCode, date_trunc('MONTH', m.EventDateTime)
        having  max(case when m.InteractionType = 'sent' then 1 else 0 end) = 1
        and     (max(case when m.InteractionType in ('open', 'click') then 1 else 0 end) = 0
                 or max(case when m.InteractionType = 'bounce' then 1 else 0 end) = 1)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R3.B1.3: Online renewal window closes at day 7


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r3_b1_3  (R3.B1.3)                                                      */
/*  Description : Renewals / Branch. Notebook 04 cell 145, corrected (D-E1.21): the online renewal wind */
/*              : ow closes at due + 7 days with no renewal by then; at due + 7, once that has passed.  */
/*              : One event per cycle.                                                                  */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyRenewal, PolicySale, RenewalPortalEven */
/*              : t                                                                                     */
/*              : Reads: stg.adsv2_renewal_base                                                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r3_b1_3 as
select  xxhash64('R3.B1.3', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R3.B1.3'                                           as EventTypeId,
        'Online renewal window closes at day 7' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(date_add(RenewalDate, 7) as timestamp)                        as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base
        where   LYPolicyTypeGroup = 'Motor'
        and     (RenewedDate is null or RenewedDate > date_add(RenewalDate, 7))
        and     date_add(RenewalDate, 7) < date '${ads.end_time}'
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R3.B1.4: Phone renewal window closes at day 30


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r3_b1_4  (R3.B1.4)                                                      */
/*  Description : Renewals / Branch. Notebook 04 cell 146, corrected (D-E1.21): the phone renewal windo */
/*              : w closes at due + 30 days with no renewal by then; at due + 30, once that has passed. */
/*              :  One event per cycle.                                                                 */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyRenewal, PolicySale, RenewalPortalEven */
/*              : t                                                                                     */
/*              : Reads: stg.adsv2_renewal_base                                                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r3_b1_4 as
select  xxhash64('R3.B1.4', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R3.B1.4'                                           as EventTypeId,
        'Phone renewal window closes at day 30' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(date_add(RenewalDate, 30) as timestamp)                       as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base
        where   LYPolicyTypeGroup = 'Motor'
        and     (RenewedDate is null or RenewedDate > date_add(RenewalDate, 30))
        and     OutcomeFinal = 1
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R3.B1.O1: Renewed inside the window


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r3_b1_o1  (R3.B1.O1)                                                    */
/*  Description : Renewals / Outcome. Notebook 04, corrected (D-E1.21): renewed after the due date but  */
/*              : inside the window, within due + 7 days online (renewal channel 'e) Online' or a porta */
/*              : l payment) or due + 30 days through the call centre; at the renewal sale date. One ev */
/*              : ent per cycle.                                                                        */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyRenewal, PolicySale, RenewalPortalEven */
/*              : t                                                                                     */
/*              : Reads: stg.adsv2_renewal_base                                                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r3_b1_o1 as
select  xxhash64('R3.B1.O1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R3.B1.O1'                                          as EventTypeId,
        'Renewed inside the window' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(RenewedDate as timestamp)                                     as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base
        where   LYPolicyTypeGroup = 'Motor'
        and     RenewedInWindow = 1
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R3.B1.O3: Lost


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r3_b1_o3  (R3.B1.O3)                                                    */
/*  Description : Renewals / Outcome. Notebook 04, corrected (D-E1.21, H10): lost = not renewed within  */
/*              : the window (due + 7 online, due + 30 call centre), from the cycle's own result; at du */
/*              : e + 30 days, once that has passed. One event per cycle.                               */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyRenewal, PolicySale, RenewalPortalEven */
/*              : t                                                                                     */
/*              : Reads: stg.adsv2_renewal_base                                                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r3_b1_o3 as
select  xxhash64('R3.B1.O3', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R3.B1.O3'                                          as EventTypeId,
        'Lost' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(date_add(RenewalDate, 30) as timestamp)                       as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base
        where   LYPolicyTypeGroup = 'Motor'
        and     Lost = 1
        and     OutcomeFinal = 1
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R6.F1: Non-arrival-  and time taken to receive


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r6_f1  (R6.F1)                                                          */
/*  Description : Renewals / Friction. Notebook 04 cell 149. The first suitability statement emailed or */
/*              :  transmitted on the new policy (not an offer).                                        */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyDocumentEvent, PolicyRenewal           */
/*              : Reads: stg.adsv2_renewal_base, stg.adsv2_relay_document_event                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r6_f1 as
select  xxhash64('R6.F1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R6.F1'                                             as EventTypeId,
        'Non-arrival-  and time taken to receive' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = b.TYPolicyCode
        where   b.LYPolicyTypeGroup = 'Motor'
        and     d.PolicyTypeGroup = 'Motor'
        and     d.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and d.EventDateTime < timestamp '${ads.end_time}'
        and     d.EventDescription like '%Suitability%'
        and     d.EventDescription not like 'Renewal Offer - %'
        and     (d.EventDescription like '%Emailed Document%' or d.EventDescription like '%Document Transmitted%')
        group by b.PolicyCode, date_trunc('MONTH', d.EventDateTime)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R4.B2.F1: Chases before the customer submits-  and documents never submitted


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r4_b2_f1  (R4.B2.F1)                                                    */
/*  Description : Renewals / Friction. Notebook 04 cell 150. Renewal motor policies with any document o */
/*              : utstanding in the chase snapshot, latest sale date per policy and month. Change: the  */
/*              : second-car certificate is tested (04 tested the column here, unlike 86-87).           */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: DocumentChaseMotorVan                        */
/*              : Reads: ods_v2.DocumentChaseMotorVan                                                   */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r4_b2_f1 as
select  xxhash64('R4.B2.F1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R4.B2.F1'                                          as EventTypeId,
        'Chases before the customer submits-  and documents never submitted' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(max(coalesce(try_cast(concat(right(SaleDate, 4), '-', substring(SaleDate, 4, 2), '-', left(SaleDate, 2)) as date), date '1900-01-01')) as timestamp)               as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.ods}.DocumentChaseMotorVan
        where   cast(coalesce(try_cast(concat(right(SaleDate, 4), '-', substring(SaleDate, 4, 2), '-', left(SaleDate, 2)) as date), date '1900-01-01') as timestamp) >= add_months(timestamp '${ads.start_time}', -1) and cast(coalesce(try_cast(concat(right(SaleDate, 4), '-', substring(SaleDate, 4, 2), '-', left(SaleDate, 2)) as date), date '1900-01-01') as timestamp) < timestamp '${ads.end_time}'
        and     PolicyType = 'Renewals'
        and     PolicyTypeGroup = 'Motor'
        and     (Sign_Prop_Status = 'O' or Sign_Stat_Of_Fact_Status = 'O' or Comp_Fin_Agree_Status = 'O' or Comp_DDM_Status = 'O' or Proof_Of_NCB_Status = 'O' or Trans_Proof_Of_NCB_Status = 'O' or Ltr_Of_Driv_Exp_Status = 'O' or Prop_Driv_Lic_Status = 'O' or Nam_Driv_Lic_Status = 'O' or Gap_In_Cov_Ltr_Status = 'O' or SecondCar_Cert_Status = 'O' or Doc_Ltr_Status = 'O' or Engineers_Rpt_Status = 'O' or NCT_Status = 'O' or Veh_Lic_Cert_Status = 'O' or Irish_Reg_Status = 'O' or Main_Driver_Dec_Status = 'O' or Soc_Dom_Pleas_Dec_Status = 'O' or Comp_Car_Exper_Status = 'O' or DD_Conf_Ltr_Status = 'O' or Comp_Fin_Mand_Status = 'O' or Val_For_Spec_Item_Status = 'O' or Dri_Lic_Num_Status = 'O' or Comp_Lost_Cert_Dec_Status = 'O' or Orig_Cert_Status = 'O' or Cancel_Req_Status = 'O' or PPS_Num_Status = 'O' or Afford_State_Status = 'O' or Identification_Status = 'O' or Digital_Journey_Status = 'O' or Finance_Form_Status = 'O')
        group by PolicyCode, date_trunc('MONTH', coalesce(try_cast(concat(right(SaleDate, 4), '-', substring(SaleDate, 4, 2), '-', left(SaleDate, 2)) as date), date '1900-01-01'))
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R4.B2.1: Document request issued


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r4_b2_1  (R4.B2.1)                                                      */
/*  Description : Renewals / Branch. Notebook 04 cell 151. A document checklist (SMS / email) saved on  */
/*              : the new policy. Change: Relay documents only (04 read every ods.EventStream source he */
/*              : re).                                                                                  */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyDocumentEvent, PolicyRenewal           */
/*              : Reads: stg.adsv2_renewal_base, stg.adsv2_relay_document_event                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r4_b2_1 as
select  xxhash64('R4.B2.1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R4.B2.1'                                           as EventTypeId,
        'Document request issued' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                b.TYPolicyCode                                                  as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = b.TYPolicyCode
        where   b.LYPolicyTypeGroup = 'Motor'
        and     d.PolicyTypeGroup = 'Motor'
        and     d.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and d.EventDateTime < timestamp '${ads.end_time}'
        and     d.EventDescription in ('New Business - Saved Document - Document Checklist SMS',
                                       'New Business - Saved Document - Document Checklist Email')
        group by b.TYPolicyCode, date_trunc('MONTH', d.EventDateTime)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R4.B2.2: Customer submits documents


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r4_b2_2  (R4.B2.2)                                                      */
/*  Description : Renewals / Branch. Notebook 04 cell 152. Document uploads on the new policy, latest p */
/*              : er policy and month.                                                                  */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: DocumentUpload, PolicyRenewal                */
/*              : Reads: stg.adsv2_renewal_base, ods_v2.DocumentUpload                                  */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r4_b2_2 as
select  xxhash64('R4.B2.2', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R4.B2.2'                                           as EventTypeId,
        'Customer submits documents' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(cast(u.EventTimestamp as timestamp))                        as EventDateTime,
                1                                                               as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.ods}.DocumentUpload u     on u.PolicyCode = b.TYPolicyCode
        where   b.LYPolicyTypeGroup = 'Motor'
        and     cast(u.EventTimestamp as timestamp) >= add_months(timestamp '${ads.start_time}', -1) and cast(u.EventTimestamp as timestamp) < add_months(timestamp '${ads.end_time}', 1)
        group by b.PolicyCode, date_trunc('MONTH', cast(u.EventTimestamp as timestamp))
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R4.B2.4: Chase-  first reminder


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r4_b2_4  (R4.B2.4)                                                      */
/*  Description : Renewals / Branch. Notebook 04 cell 153. Renewal motor policies with any document out */
/*              : standing: the first chase record per policy and month.                                */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: DocumentChaseMotorVan                        */
/*              : Reads: ods_v2.DocumentChaseMotorVan                                                   */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r4_b2_4 as
select  xxhash64('R4.B2.4', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R4.B2.4'                                           as EventTypeId,
        'Chase-  first reminder' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(min(CreateDate) as timestamp)                              as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.ods}.DocumentChaseMotorVan
        where   cast(coalesce(try_cast(concat(right(SaleDate, 4), '-', substring(SaleDate, 4, 2), '-', left(SaleDate, 2)) as date), date '1900-01-01') as timestamp) >= add_months(timestamp '${ads.start_time}', -1) and cast(coalesce(try_cast(concat(right(SaleDate, 4), '-', substring(SaleDate, 4, 2), '-', left(SaleDate, 2)) as date), date '1900-01-01') as timestamp) < timestamp '${ads.end_time}'
        and     PolicyType = 'Renewals'
        and     PolicyTypeGroup = 'Motor'
        and     (Sign_Prop_Status = 'O' or Sign_Stat_Of_Fact_Status = 'O' or Comp_Fin_Agree_Status = 'O' or Comp_DDM_Status = 'O' or Proof_Of_NCB_Status = 'O' or Trans_Proof_Of_NCB_Status = 'O' or Ltr_Of_Driv_Exp_Status = 'O' or Prop_Driv_Lic_Status = 'O' or Nam_Driv_Lic_Status = 'O' or Gap_In_Cov_Ltr_Status = 'O' or SecondCar_Cert_Status = 'O' or Doc_Ltr_Status = 'O' or Engineers_Rpt_Status = 'O' or NCT_Status = 'O' or Veh_Lic_Cert_Status = 'O' or Irish_Reg_Status = 'O' or Main_Driver_Dec_Status = 'O' or Soc_Dom_Pleas_Dec_Status = 'O' or Comp_Car_Exper_Status = 'O' or DD_Conf_Ltr_Status = 'O' or Comp_Fin_Mand_Status = 'O' or Val_For_Spec_Item_Status = 'O' or Dri_Lic_Num_Status = 'O' or Comp_Lost_Cert_Dec_Status = 'O' or Orig_Cert_Status = 'O' or Cancel_Req_Status = 'O' or PPS_Num_Status = 'O' or Afford_State_Status = 'O' or Identification_Status = 'O' or Digital_Journey_Status = 'O' or Finance_Form_Status = 'O')
        group by PolicyCode, date_trunc('MONTH', CreateDate)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R4.B2.5: Chase-  escalation


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r4_b2_5  (R4.B2.5)                                                      */
/*  Description : Renewals / Branch. Notebook 04 cell 154. A final chase document on the new policy (an */
/*              : y product, as 04).                                                                    */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyDocumentEvent, PolicyRenewal           */
/*              : Reads: stg.adsv2_renewal_base, stg.adsv2_relay_document_event                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r4_b2_5 as
select  xxhash64('R4.B2.5', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R4.B2.5'                                           as EventTypeId,
        'Chase-  escalation' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = b.TYPolicyCode
        where   b.LYPolicyTypeGroup = 'Motor'
        and     d.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and d.EventDateTime < add_months(timestamp '${ads.end_time}', 1)
        and     d.EventDescription like '%Chase%'
        and     d.EventDescription like '%Final%'
        group by b.PolicyCode, date_trunc('MONTH', d.EventDateTime)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R4.B2.3: Documents validated


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r4_b2_3  (R4.B2.3)                                                      */
/*  Description : Renewals / Branch. Notebook 04 cell 155. Accepted documents on the new policy in the  */
/*              : window, latest per policy and month.                                                  */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: DocumentWorkflowStatus, PolicyRenewal        */
/*              : Reads: stg.adsv2_renewal_base, ods_v2.DocumentWorkflowStatus                          */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r4_b2_3 as
select  xxhash64('R4.B2.3', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R4.B2.3'                                           as EventTypeId,
        'Documents validated' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(cast(w.EventTimestamp as timestamp))                        as EventDateTime,
                1                                                               as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.ods}.DocumentWorkflowStatus w on w.PolicyCode = b.TYPolicyCode
        where   b.LYPolicyTypeGroup = 'Motor'
        and     cast(w.EventTimestamp as timestamp) >= timestamp '${ads.start_time}' and cast(w.EventTimestamp as timestamp) < timestamp '${ads.end_time}'
        and     w.IsAccepted = true
        group by b.PolicyCode, date_trunc('MONTH', cast(w.EventTimestamp as timestamp))
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R4.B2.O1: Cancellation-  non-receipt


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r4_b2_o1  (R4.B2.O1)                                                    */
/*  Description : Renewals / Outcome. Notebook 04 cell 156. An insurer-led cancellation email on a new  */
/*              : policy that also had a final chase.                                                   */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyDocumentEvent, PolicyRenewal           */
/*              : Reads: stg.adsv2_renewal_base, stg.adsv2_relay_document_event                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r4_b2_o1 as
select  xxhash64('R4.B2.O1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R4.B2.O1'                                          as EventTypeId,
        'Cancellation-  non-receipt' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = b.TYPolicyCode
        left semi join (
                select  PolicyCode
                from    ${ads.stg}.adsv2_relay_document_event
                where   EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and EventDateTime < add_months(timestamp '${ads.end_time}', 1)
                and     EventDescription like '%Chase%'
                and     EventDescription like '%Final%'
                ) c     on c.PolicyCode = b.TYPolicyCode
        where   b.LYPolicyTypeGroup = 'Motor'
        and     d.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and d.EventDateTime < add_months(timestamp '${ads.end_time}', 1)
        and     d.EventDescription in ('Insurer Led Cancelation - Emailed Document - Reg canx email template',
                                       'Insurer Led Cancelation - Sent To Document Processing - Reg canx email template')
        group by b.PolicyCode, date_trunc('MONTH', d.EventDateTime)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R4.B2.6: Documents received and validated


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r4_b2_6  (R4.B2.6)                                                      */
/*  Description : Renewals / Branch. Notebook 04 cell 157. The same rule as R4.B2.3 (as today).         */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: DocumentWorkflowStatus, PolicyRenewal        */
/*              : Reads: stg.adsv2_renewal_base, ods_v2.DocumentWorkflowStatus                          */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r4_b2_6 as
select  xxhash64('R4.B2.6', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R4.B2.6'                                           as EventTypeId,
        'Documents received and validated' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(cast(w.EventTimestamp as timestamp))                        as EventDateTime,
                1                                                               as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.ods}.DocumentWorkflowStatus w on w.PolicyCode = b.TYPolicyCode
        where   b.LYPolicyTypeGroup = 'Motor'
        and     cast(w.EventTimestamp as timestamp) >= timestamp '${ads.start_time}' and cast(w.EventTimestamp as timestamp) < timestamp '${ads.end_time}'
        and     w.IsAccepted = true
        group by b.PolicyCode, date_trunc('MONTH', cast(w.EventTimestamp as timestamp))
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R6.B1: Cert and disc dispatch


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r6_b1  (R6.B1)                                                          */
/*  Description : Renewals / Branch. Notebook 04 cell 158. Any document emailed or transmitted on the n */
/*              : ew policy of a retained renewal (not an offer or a chase). Change: Relay documents on */
/*              : ly (04 read every ods.EventStream source here).                                       */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyDocumentEvent, PolicyRenewal           */
/*              : Reads: stg.adsv2_renewal_base, stg.adsv2_relay_document_event                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r6_b1 as
select  xxhash64('R6.B1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R6.B1'                                             as EventTypeId,
        'Cert and disc dispatch' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = b.TYPolicyCode
        where   b.LYPolicyTypeGroup = 'Motor'
        and     b.PolicyRetNum = 1
        and     d.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and d.EventDateTime < timestamp '${ads.end_time}'
        and     (d.EventDescription like '%Emailed Document%' or d.EventDescription like '%Document Transmitted%')
        and     d.EventDescription not like 'Renewal Offer%'
        and     d.EventDescription not like 'Document Chase%'
        group by b.PolicyCode, date_trunc('MONTH', d.EventDateTime)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R6.B1.1: Cert and disc dispatched


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r6_b1_1  (R6.B1.1)                                                      */
/*  Description : Renewals / Branch. Notebook 04 cell 159. The same rule as R6.B1 (as today).           */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyDocumentEvent, PolicyRenewal           */
/*              : Reads: stg.adsv2_renewal_base, stg.adsv2_relay_document_event                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r6_b1_1 as
select  xxhash64('R6.B1.1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R6.B1.1'                                           as EventTypeId,
        'Cert and disc dispatched' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = b.TYPolicyCode
        where   b.LYPolicyTypeGroup = 'Motor'
        and     b.PolicyRetNum = 1
        and     d.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and d.EventDateTime < timestamp '${ads.end_time}'
        and     (d.EventDescription like '%Emailed Document%' or d.EventDescription like '%Document Transmitted%')
        and     d.EventDescription not like 'Renewal Offer%'
        and     d.EventDescription not like 'Document Chase%'
        group by b.PolicyCode, date_trunc('MONTH', d.EventDateTime)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R6.B1.2: Document pack dispatched


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r6_b1_2  (R6.B1.2)                                                      */
/*  Description : Renewals / Branch. Notebook 04 cell 160. The same rule as R6.B1 (as today).           */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyDocumentEvent, PolicyRenewal           */
/*              : Reads: stg.adsv2_renewal_base, stg.adsv2_relay_document_event                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r6_b1_2 as
select  xxhash64('R6.B1.2', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R6.B1.2'                                           as EventTypeId,
        'Document pack dispatched' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = b.TYPolicyCode
        where   b.LYPolicyTypeGroup = 'Motor'
        and     b.PolicyRetNum = 1
        and     d.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and d.EventDateTime < timestamp '${ads.end_time}'
        and     (d.EventDescription like '%Emailed Document%' or d.EventDescription like '%Document Transmitted%')
        and     d.EventDescription not like 'Renewal Offer%'
        and     d.EventDescription not like 'Document Chase%'
        group by b.PolicyCode, date_trunc('MONTH', d.EventDateTime)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## R6.B1.5: Replacement requested


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_r6_b1_5  (R6.B1.5)                                                      */
/*  Description : Renewals / Branch. Notebook 04 cell 161. A duplicate certificate on the new policy (r */
/*              : eplacement requested).                                                                */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyDocumentEvent, PolicyRenewal           */
/*              : Reads: stg.adsv2_renewal_base, stg.adsv2_relay_document_event                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_r6_b1_5 as
select  xxhash64('R6.B1.5', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'R6.B1.5'                                           as EventTypeId,
        'Replacement requested' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = b.TYPolicyCode
        where   b.LYPolicyTypeGroup = 'Motor'
        and     d.PolicyTypeGroup = 'Motor'
        and     d.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and d.EventDateTime < timestamp '${ads.end_time}'
        and     d.EventDescription like 'Duplicate Certificate%'
        group by b.PolicyCode, date_trunc('MONTH', d.EventDateTime)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

In [ ]:
%%pyspark
# Events per family (a family with no events is reported, not failed: check its source and window)
for t in ["adsv2_evt_r0", "adsv2_evt_r0_f1", "adsv2_evt_r0_b1", "adsv2_evt_r0_f2", "adsv2_evt_r1a", "adsv2_evt_r1b", "adsv2_evt_r1_b2", "adsv2_evt_r1_e1", "adsv2_evt_r1_e1_f3", "adsv2_evt_r2a", "adsv2_evt_r2b", "adsv2_evt_r2_f1", "adsv2_evt_r2_b1_f1", "adsv2_evt_r2_e1", "adsv2_evt_r3a", "adsv2_evt_r3b", "adsv2_evt_r3c", "adsv2_evt_r3d", "adsv2_evt_r3b_f1", "adsv2_evt_r3_o1", "adsv2_evt_r4", "adsv2_evt_r4_f1", "adsv2_evt_r4a", "adsv2_evt_r4c", "adsv2_evt_r5", "adsv2_evt_r6", "adsv2_evt_r2_b1_1", "adsv2_evt_r2_b1_2", "adsv2_evt_r2_b1_3", "adsv2_evt_r2_b1_3_f1", "adsv2_evt_r2_b1_4_f1", "adsv2_evt_r2_b1_o1", "adsv2_evt_r2_b1_o2", "adsv2_evt_r2_b1_o3", "adsv2_evt_r3_b1", "adsv2_evt_r3_b1_1", "adsv2_evt_r3_b1_1_f1", "adsv2_evt_r3_b1_2", "adsv2_evt_r3_b1_2_f1", "adsv2_evt_r3_b1_3", "adsv2_evt_r3_b1_4", "adsv2_evt_r3_b1_o1", "adsv2_evt_r3_b1_o3", "adsv2_evt_r6_f1", "adsv2_evt_r4_b2_f1", "adsv2_evt_r4_b2_1", "adsv2_evt_r4_b2_2", "adsv2_evt_r4_b2_4", "adsv2_evt_r4_b2_5", "adsv2_evt_r4_b2_3", "adsv2_evt_r4_b2_o1", "adsv2_evt_r4_b2_6", "adsv2_evt_r6_b1", "adsv2_evt_r6_b1_1", "adsv2_evt_r6_b1_2", "adsv2_evt_r6_b1_5"]:
    n = spark.table(f"{stg_schema}.{t}").count()
    print(f"{t:<40} {n:>12,}" + ("   <- no events" if n == 0 else ""))
ads_run_log("12_Motor_Renewal", "succeeded")
